# Reference  
I have referred to many existing notebooks. I would like to thank everyone.  
- https://www.kaggle.com/code/ryanholbrook/getting-started-with-santa-2022  
- https://www.kaggle.com/code/crodoc/82409-improved-baseline-santa-2022
- https://www.kaggle.com/code/nicupetridean/further-analysis-of-costs-points-ordering  
- https://www.kaggle.com/code/snufkin77/further-point-order-improvements-tsp-start  
- https://www.kaggle.com/code/elvenmonk/santa-2022-lower-bound-approximation-73078

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from functools import *
from itertools import *
from pathlib import Path
from PIL import Image
from tqdm import tqdm

plt.style.use('seaborn-whitegrid')

data_dir = Path('/kaggle/input/santa-2022')

df_image = pd.read_csv(data_dir / 'image.csv')

df_image

# Helper Function  

In [ ]:
def draw_path(points, strategy='printing'):
    '''DUMB idea to plot the strategy path to print the card''' 
    points_list = list()
    for (i,j) in points:
        points_list.append([i,j])

    data = np.array(points_list)

    # plotting a line plot after changing it's width and height
    f = plt.figure()
    f.set_figwidth(15)
    f.set_figheight(15)

    print(f"Plotting the {strategy} approach")
    plt.plot(data[:, 0], data[:, 1])
    plt.show()
    
# Functions to map between cartesian coordinates and array indexes
def cartesian_to_array(x, y, shape):
    m, n = shape[:2]
    i = (n - 1) // 2 - y
    j = (n - 1) // 2 + x
    if i < 0 or i >= m or j < 0 or j >= n:
        raise ValueError("Coordinates not within given dimensions.")
    return i, j


def array_to_cartesian(i, j, shape):
    m, n = shape[:2]
    if i < 0 or i >= m or j < 0 or j >= n:
        raise ValueError("Coordinates not within given dimensions.")
    y = (n - 1) // 2 - i
    x = j - (n - 1) // 2
    return x, y


point = (1, 8)
shape = (9, 9, 3)
assert cartesian_to_array(*array_to_cartesian(*point, shape), shape) == point


# Functions to map an image between array and record formats
def image_to_dict(image):
    image = np.atleast_3d(image)
    kv_image = {}
    for i, j in product(range(len(image)), repeat=2):
        kv_image[array_to_cartesian(i, j, image.shape)] = tuple(image[i, j])
    return kv_image


def image_to_df(image):
    return pd.DataFrame(
        [(x, y, r, g, b) for (x, y), (r, g, b) in image_to_dict(image).items()],
        columns=['x', 'y', 'r', 'g', 'b']
    )


def df_to_image(df):
    side = int(len(df) ** 0.5)  # assumes a square image
    return df.set_index(['x', 'y']).to_numpy().reshape(side, side, -1)

def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

def rotate_link(vector, direction):
    x, y = vector
    if direction == 1:  # counter-clockwise
        if y >= x and y > -x:
            x -= 1
        elif y > x and y <= -x:
            y -= 1
        elif y <= x and y < -x:
            x += 1
        else:
            y += 1
    elif direction == -1:  # clockwise
        if y > x and y >= -x:
            x += 1
        elif y >= x and y < -x:
            y += 1
        elif y < x and y <= -x:
            x -= 1
        else:
            y -= 1
    return (x, y)


def rotate(config, i, direction):
    config = config.copy()
    config[i] = rotate_link(config[i], direction)
    return config


def get_square(link_length):
    link = (link_length, 0)
    coords = [link]
    for _ in range(8 * link_length - 1):
        link = rotate_link(link, direction=1)
        coords.append(link)
    return coords

def get_neighbors(config):
    nhbrs = (
        reduce(lambda x, y: rotate(x, *y), enumerate(directions), config)
        for directions in product((-1, 0, 1), repeat=len(config))
    )
    return list(filter(lambda c: c != config, nhbrs))

# Functions to compute the cost function

# Cost of reconfiguring the robotic arm: the square root of the number of links rotated
def reconfiguration_cost(from_config, to_config):
    nlinks = len(from_config)
    diffs = np.abs(np.asarray(from_config) - np.asarray(to_config)).sum(axis=1)
    return np.sqrt(diffs.sum())


# Cost of moving from one color to another: the sum of the absolute change in color components
def color_cost(from_position, to_position, image, color_scale=3.0):
    return np.abs(image[to_position] - image[from_position]).sum() * color_scale


# Total cost of one step: the reconfiguration cost plus the color cost
def step_cost(from_config, to_config, image):
    from_position = cartesian_to_array(*get_position(from_config), image.shape)
    to_position = cartesian_to_array(*get_position(to_config), image.shape)
    return (
        reconfiguration_cost(from_config, to_config) +
        color_cost(from_position, to_position, image)
    )

In [ ]:
def get_direction(u, v):
    """Returns the sign of the angle from u to v."""
    direction = np.sign(np.cross(u, v))
    if direction == 0 and np.dot(u, v) < 0:
        direction = 1
    return direction


# We don't use this elsewhere, but you might find it useful."""
def get_angle(u, v):
    """Returns the angle (in degrees) from u to v."""
    return np.degrees(np.math.atan2(
        np.cross(u, v),
        np.dot(u, v),
    ))


def get_path_to_point(config, point):
    """Find a path of configurations to `point` starting at `config`."""
    path = [config]
    # Rotate each link, starting with the largest, until the point can
    # be reached by the remaining links. The last link must reach the
    # point itself.
    for i in range(len(config)):
        link = config[i]
        base = get_position(config[:i])
        relbase = (point[0] - base[0], point[1] - base[1])
        position = get_position(config[:i+1])
        relpos = (point[0] - position[0], point[1] - position[1])
        radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
        # Special case when next-to-last link lands on point.
        if radius == 1 and relpos == (0, 0):
            config = rotate(config, i, 1)
            if get_position(config) == point:  # Thanks @pgeiger
                path.append(config)
                break
            else:
                continue
        while np.max(np.abs(relpos)) > radius:
            direction = get_direction(link, relbase)
            config = rotate(config, i, direction)
            path.append(config)
            link = config[i]
            base = get_position(config[:i])
            relbase = (point[0] - base[0], point[1] - base[1])
            position = get_position(config[:i+1])
            relpos = (point[0] - position[0], point[1] - position[1])
            radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
    assert get_position(path[-1]) == point
    return path

def get_path_to_configuration(from_config, to_config):
    path = [from_config]
    config = from_config.copy()
    while config != to_config:
        for i in range(len(config)):
            config = rotate(config, i, get_direction(config[i], to_config[i]))
        path.append(config)
    assert path[-1] == to_config
    return path

# Compute total cost of path over image
def total_cost(path, image):
    return reduce(
        lambda cost, pair: cost + step_cost(pair[0], pair[1], image),
        zip(path[:-1], path[1:]),
        0,
    )

In [ ]:
# compress a path between two points
def compress_path(path):
    r = [[] for _ in range(8)]
    for p in path:
        for i in range(8):
            if len(r[i]) == 0 or r[i][-1] != p[i]:
                r[i].append(p[i])
    mx = max([len(x) for x in r])
    
    for rr in r:
        while len(rr) < mx:
            rr.append(rr[-1])
    r = list(zip(*r))
    for i in range(len(r)):
        r[i] = list(r[i])
    return r

def get_path_to_point(config, point):
    """Find a path of configurations to `point` starting at `config`."""
    path = [config]
    # Rotate each link, starting with the largest, until the point can
    # be reached by the remaining links. The last link must reach the
    # point itself.
    for i in range(len(config)):
        link = config[i]
        base = get_position(config[:i])
        relbase = (point[0] - base[0], point[1] - base[1])
        position = get_position(config[:i+1])
        relpos = (point[0] - position[0], point[1] - position[1])
        radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
        # Special case when next-to-last link lands on point. 
        if radius == 1 and relpos == (0, 0):
            config = rotate(config, i, 1)
            if get_position(config) == point:
                path.append(config)
                break
            else:
                continue
        while np.max(np.abs(relpos)) > radius:
            direction = get_direction(link, relbase)
            config = rotate(config, i, direction)
            path.append(config)
            link = config[i]
            base = get_position(config[:i])
            relbase = (point[0] - base[0], point[1] - base[1])
            position = get_position(config[:i+1])
            relpos = (point[0] - position[0], point[1] - position[1])
            radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
    assert get_position(path[-1]) == point
    path = compress_path(path)
    return path

# Make Points  
By default, this code uses a [hill climbing](https://en.wikipedia.org/wiki/Hill_climbing) methodology.  
1. Calculates costs for navigable surrounding locations.  
2. Move to the lowest cost location.  
  - If the new location is the one we visited before, move to the next one with less cost.  
  
Add the following steps to prevent falling into local minima and to add exploratory navigation elements.  
- Before deciding the next location in cost order, delete some of the smaller cost locations with a specific probability.  
  - The current code will delete the top 5% candidates with a 5% chance.  

In [ ]:
import random

image = df_to_image(df_image)
size = image.shape[0]
closest = np.zeros((size, size)) # for checking visit

x, y = 0, 0
s_r = 32 # search range

test_points = []
closest[x, y] = 1
t_cost = 0
for step in tqdm(range(size * size)):
    test_points.append((x-128, y-128))
    
    cur_image = image[x, y, :]
    dx_list, dy_list, c_list = [], [], []
    for dx in range(-s_r, s_r+1):
        for dy in range(-s_r, s_r+1):
            if abs(dx) + abs(dy) <= s_r and abs(dx) + abs(dy) >0\
            and x + dx >= 0 and x + dx < size\
            and y + dy >= 0 and y + dy < size:
                tar_image = image[x+dx, y+dy, :]
                cost = np.sqrt(abs(dx) + abs(dy)) - 1 + (np.abs(tar_image - cur_image).sum()*3)
                dx_list.append(dx)
                dy_list.append(dy)
                c_list.append(cost)
                
    temp_ = pd.DataFrame({'dx':dx_list, 'dy':dy_list, 'cost':c_list}).sort_values(by=['cost']).reset_index(drop=True)
    
    # explore
    random.seed(step)
    r_ = random.uniform(1, 100)
    prob_ = 0.05
    if r_ <= 100*prob_: 
        temp_ = temp_.iloc[int(prob_*len(temp_)):]
    
    if len(temp_) > 0:
        flag = False # available go next step
        for i, row in temp_.iterrows():
            new_x = int(x + row['dx'])
            new_y = int(y + row['dy'])
            
            if closest[new_x, new_y]==0:
                closest[new_x, new_y] = 1
                x = new_x
                y = new_y
                flag = True
                t_cost += row['cost']
                break
            else:
                # current new_x, new_y is already visited
                continue
        
        if not flag:
            print(f"Can't Go Next Step : {step, new_x, new_y}")
            break
    else:
        print("There is no candidate")

## Check Result  
It has stopped at 95% of the total progress.  
This is because all locations within the search range are locations that we have already visited before.  

In [ ]:
origin = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]
# n = origin[0][0] * 2

#Add Step Cost
#Add Neighbors
visited = []
path = [origin]
for p in tqdm(test_points):
    config = path[-1]
    if p not in visited:
        candy_cane_road = get_path_to_point(config, p)[1:]
        if len(candy_cane_road)>0:
            visited += [get_position(r) for r in candy_cane_road]
        path.extend(candy_cane_road)
path.extend(get_path_to_configuration(path[-1], origin)[1:])
total_cost(path, image)

We've only visited 95% of the locations, but the total cost is already quite high.  
Due to the random navigation elements, a number of large positional variations are also found in the figure below.  
A big change in position to the next step has a very bad effect on total cost.  

In [ ]:
draw_path(test_points, 'greed-approach')

# Additional 
If there is no exploratory navigation elements?  
This is simply the application of the heel climbing methodology.  

In [ ]:
import random

image = df_to_image(df_image)
size = image.shape[0]
closest = np.zeros((size, size)) # for checking visit

x, y = 0, 0
s_r = 32 # search range

test_points = []
closest[x, y] = 1
t_cost = 0
for step in tqdm(range(size * size)):
    test_points.append((x-128, y-128))
    
    cur_image = image[x, y, :]
    dx_list, dy_list, c_list = [], [], []
    for dx in range(-s_r, s_r+1):
        for dy in range(-s_r, s_r+1):
            if abs(dx) + abs(dy) <= s_r and abs(dx) + abs(dy) >0\
            and x + dx >= 0 and x + dx < size\
            and y + dy >= 0 and y + dy < size:
                tar_image = image[x+dx, y+dy, :]
                cost = np.sqrt(abs(dx) + abs(dy)) - 1 + (np.abs(tar_image - cur_image).sum()*3)
                dx_list.append(dx)
                dy_list.append(dy)
                c_list.append(cost)
                
    temp_ = pd.DataFrame({'dx':dx_list, 'dy':dy_list, 'cost':c_list}).sort_values(by=['cost']).reset_index(drop=True)
    
#     # explore
#     random.seed(step)
#     r_ = random.uniform(1, 100)
#     prob_ = 0.05
#     if r_ <= 100*prob_: 
#         temp_ = temp_.iloc[int(prob_*len(temp_)):]
    
    if len(temp_) > 0:
        flag = False # available go next step
        for i, row in temp_.iterrows():
            new_x = int(x + row['dx'])
            new_y = int(y + row['dy'])
            
            if closest[new_x, new_y]==0:
                closest[new_x, new_y] = 1
                x = new_x
                y = new_y
                flag = True
                t_cost += row['cost']
                break
            else:
                # current new_x, new_y is already visited
                continue
        
        if not flag:
            print(f"Can't Go Next Step : {step, new_x, new_y}")
            break
    else:
        print("There is no candidate")

In [ ]:
origin = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]
# n = origin[0][0] * 2

#Add Step Cost
#Add Neighbors
visited = []
path = [origin]
for p in tqdm(test_points):
    config = path[-1]
    if p not in visited:
        candy_cane_road = get_path_to_point(config, p)[1:]
        if len(candy_cane_road)>0:
            visited += [get_position(r) for r in candy_cane_road]
        path.extend(candy_cane_road)
path.extend(get_path_to_configuration(path[-1], origin)[1:])
total_cost(path, image)

If there were no random exploration elements, 79% of the locations were visited.  
In addition, the total cost is high even though it is 79%.  
If you look at the picture below, you can see a lot of empty spots, although the change in position movement has decreased significantly.  
Also, in the picture below, we can see a faint outline of Rudolph, the outline of the tree, the outline of the snowman's torso.  

In [ ]:
draw_path(test_points, 'greed-approach')

# Conclusion  
By default, we should find the path that travels the shortest distance with less color change.  
In this respect, there is a significant risk to the greedy search.  
We can change positions as jump (like Random Explore), but this dramatically increases the total cost.  
It may be efficient to navigate only the regions of the partial elements on the image.  